In [ ]:
model_path = "meta-llama/Llama-2-7b-chat-hf"
device = "cuda:0"
prompt_tmp = """Solve the following math problem step by step. 
At the end, give the final answer with only the number.

Problem:
{question}
"""

In [ ]:
import torch
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(model_path)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    dtype=torch.bfloat16,
    device_map=device,
    )

In [ ]:
from datasets import load_dataset

test_data = load_dataset("openai/gsm8k", "main",split="test")
train_data = load_dataset("openai/gsm8k", "main",split="train")

In [ ]:
import re
import time
def find_answer(str):
    ans = re.findall(r"-?\d[\d,.]*", str)
    if not ans:
        print(str)
    while ans[-1][-1] == '.':
        ans[-1] = ans[-1][:-1]
    return float(ans[-1].replace(",","")) if ans else None

def generate(model, Q):
    prompt = prompt_tmp.format(question=Q)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    outputs = model.generate(**inputs,max_new_tokens=512,repetition_penalty=1.2,do_sample=True,temperature=0.7,
        eos_token_id=tokenizer.eos_token_id,pad_token_id=tokenizer.eos_token_id)
    
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return text

def test_model(num):
    global model
    correct = 0
    try:
        num = int(num)
        if num <=0 or num > len(test_data):
            num = 100
    except:
        num = 100
    
    start = time.time()
    for i in range(num):
        item = test_data[i]
        Q = item['question']
        A = find_answer(item['answer'])
        result = generate(model,Q)
        model_A = find_answer(result)
        #print(f"***{i}***\n",result,"\n\nAAAA = ",A,"aaaa = ",model_A)
        if A == model_A:
            correct += 1
    end = time.time()
    performance = {
        "time": end - start,
        "accuracy": correct / num * 100,
        "num" : num
    }
    return performance

In [ ]:
test_result = test_model(10)

In [ ]:
print(test_result)